# Data Check

Check which inference result files currently exist against all possible experiment combinations defined in `scripts/submit_job.sh`.

Covers:
- **Baseline** runs: `output/full_baseline/{model}/{dataset}_greedy_baseline.json`
- **Two-pass** runs: `output/full_transition/{model}/{dataset}_greedy_mt{tokens}_{suffix}.json`

In [16]:
"""All experiment combinations mirrored from scripts/submit_job.sh."""

import json
from pathlib import Path

# notebooks run from notebooks/, so step up to reach the repo root
OUTPUT_DIR = Path("..") / "output"

# models array from submit_job.sh
MODELS = [
    "qwen3-8b",
    "olmo-3-7b-think",
    "code-nemotron-7b",
]

# eval_datasets array from submit_job.sh
DATASETS = [
    "code/evalplus",
    "code/livecodebench",
    "code/bigcodebench",
    "code/editbench",
    "code/codereval",
    "crux/cruxeval_i",
    "crux/cruxeval_o",
    "reasoning/gpqa",
    "math/gsm8k",
    "math/math500",
]

# inference config profile — fixed across all runs
CONFIG = "greedy"

# max_think_tokens sweep values from submit_job.sh (includes commented-out entries)
MAX_THINK_TOKENS = [
    4096,
    8192,
    12288,
    16384,
    20480,
    24576,
    28664,
]

# overflow suffix keys from submit_job.sh ("blank" excluded — not a valid key in the codebase)
OVERFLOW_SUFFIXES = [
    "base",
    # "truncated",
    # "formal",
    # "human",
]

# these datasets require a separate execution-based evaluation pipeline;
# the summary records pass_at_1=0.0 as a placeholder until that is run
EXTERNALLY_EVALUATED = {"bigcodebench", "editbench", "codereval"}

In [17]:
"""Check all baseline result files and summary keys (one per model × dataset combination)."""

# column widths derived from the data
COL_D = max(len(d) for d in DATASETS) + 2
COL_S = 7  # matches "summary" header

# track totals for the summary at the end
baseline_found = 0
baseline_total = 0
# separately track how many baseline runs have been evaluated
baseline_evaluated = 0

for model in MODELS:
    # load the per-model summary JSON once — keys are "{dataset}_{config}_{run_name}"
    summary_path = OUTPUT_DIR / f"{model}.json"
    summary = json.load(open(summary_path)) if summary_path.exists() else {}

    # collect results for this model first so the counts can go in the header
    model_rows = []
    for dataset in DATASETS:
        # dataset stem is the final path component (e.g. "code/evalplus" → "evalplus")
        stem = Path(dataset).stem
        file_path = (
            OUTPUT_DIR / "full_baseline" / model / f"{stem}_{CONFIG}_baseline.json"
        )
        file_exists = file_path.exists()
        summary_key = f"{dataset}_{CONFIG}_baseline"
        summary_exists = summary_key in summary

        # evaluated = summary exists for most datasets; for externally-evaluated ones
        # the summary placeholder is 0.0, so we only tick ✓ once a real score is present
        if stem in EXTERNALLY_EVALUATED:
            evaluated = summary_exists and summary[summary_key].get("pass_at_1", 0) > 0
        else:
            evaluated = summary_exists

        model_rows.append((dataset, file_exists, summary_exists, evaluated))
        baseline_total += 1
        baseline_found += int(file_exists)
        baseline_evaluated += int(evaluated)

    f_count = sum(int(f) for _, f, _, _ in model_rows)
    s_count = sum(int(s) for _, _, s, _ in model_rows)
    e_count = sum(int(e) for _, _, _, e in model_rows)

    print(
        f"── {model} ── "
        f"(file: {f_count}/{len(DATASETS)}, summary: {s_count}/{len(DATASETS)}, evaluated: {e_count}/{len(DATASETS)})"
    )
    print(f"  {'dataset':{COL_D}}  {'file':{COL_S}}  {'summary':{COL_S}}  evaluated")
    print(f"  {'─' * COL_D}  {'─' * COL_S}  {'─' * COL_S}  {'─' * COL_S}")

    for dataset, file_exists, summary_exists, evaluated in model_rows:
        f_tick = "✓" if file_exists else "✗"
        s_tick = "✓" if summary_exists else "✗"
        # mark externally-evaluated datasets that are not yet done with "ext"
        stem = Path(dataset).stem
        if stem in EXTERNALLY_EVALUATED and not evaluated:
            e_tick = "ext"
        else:
            e_tick = "✓" if evaluated else "✗"
        print(f"  {dataset:{COL_D}}  {f_tick:{COL_S}}  {s_tick:{COL_S}}  {e_tick}")

    print()

── qwen3-8b ── (file: 10/10, summary: 10/10, evaluated: 7/10)
  dataset               file     summary  evaluated
  ────────────────────  ───────  ───────  ───────
  code/evalplus         ✓        ✓        ✓
  code/livecodebench    ✓        ✓        ✓
  code/bigcodebench     ✓        ✓        ext
  code/editbench        ✓        ✓        ext
  code/codereval        ✓        ✓        ext
  crux/cruxeval_i       ✓        ✓        ✓
  crux/cruxeval_o       ✓        ✓        ✓
  reasoning/gpqa        ✓        ✓        ✓
  math/gsm8k            ✓        ✓        ✓
  math/math500          ✓        ✓        ✓

── olmo-3-7b-think ── (file: 10/10, summary: 1/10, evaluated: 0/10)
  dataset               file     summary  evaluated
  ────────────────────  ───────  ───────  ───────
  code/evalplus         ✓        ✗        ✗
  code/livecodebench    ✓        ✗        ✗
  code/bigcodebench     ✓        ✗        ext
  code/editbench        ✓        ✗        ext
  code/codereval        ✓        ✓     

In [18]:
"""Check all two-pass result files and summary keys (model × dataset × max_think_tokens × overflow_suffix)."""

# column widths derived from the data
COL_R = max(len(f"mt{t}_{s}") for t in MAX_THINK_TOKENS for s in OVERFLOW_SUFFIXES) + 2
COL_S = max(len("partial (10/10)"), len("complete")) + 1


# helper: compact status label
def _status(n: int, total: int) -> str:
    if n == total:
        return "complete"
    if n == 0:
        return "missing"
    return f"partial ({n}/{total})"


# track totals for the summary at the end
twopass_found = 0
twopass_total = 0

model_combos = len(DATASETS) * len(MAX_THINK_TOKENS) * len(OVERFLOW_SUFFIXES)

for model in MODELS:
    # load the per-model summary JSON once
    summary_path = OUTPUT_DIR / f"{model}.json"
    summary = json.load(open(summary_path)) if summary_path.exists() else {}

    # collect all run results for this model first so the counts can go in the header
    model_runs = []
    model_found = 0

    for tokens in MAX_THINK_TOKENS:
        for suffix in OVERFLOW_SUFFIXES:
            # run name encodes the token cap and suffix key
            run_name = f"mt{tokens}_{suffix}"

            n_file = 0
            n_summary = 0
            missing_datasets = []

            for dataset in DATASETS:
                stem = Path(dataset).stem
                path = (
                    OUTPUT_DIR
                    / "full_transition"
                    / model
                    / f"{stem}_{CONFIG}_{run_name}.json"
                )
                file_exists = path.exists()
                summary_exists = f"{dataset}_{CONFIG}_{run_name}" in summary

                n_file += int(file_exists)
                n_summary += int(summary_exists)
                if not file_exists:
                    missing_datasets.append(dataset)

            model_found += n_file
            model_runs.append((run_name, n_file, n_summary, missing_datasets))

    s_total = sum(s for _, _, s, _ in model_runs)
    n = len(DATASETS)

    print(
        f"── {model} ── (file: {model_found}/{model_combos}, summary: {s_total}/{model_combos})"
    )
    print(f"  {'run':{COL_R}}  {'file':{COL_S}}  summary")
    print(f"  {'─' * COL_R}  {'─' * COL_S}  {'─' * COL_S}")

    for run_name, n_file, n_summary, missing_datasets in model_runs:
        print(
            f"  {run_name:{COL_R}}  {_status(n_file, n):{COL_S}}  {_status(n_summary, n)}"
        )

        # list individual missing datasets when the run is partial
        if 0 < n_file < n:
            for d in missing_datasets:
                print(f"    ✗  {d}")

    twopass_total += model_combos
    twopass_found += model_found
    print()

── qwen3-8b ── (file: 65/70, summary: 62/70)
  run             file              summary
  ──────────────  ────────────────  ────────────────
  mt4096_base     complete          complete
  mt8192_base     complete          complete
  mt12288_base    complete          complete
  mt16384_base    complete          complete
  mt20480_base    complete          complete
  mt24576_base    partial (9/10)    partial (9/10)
    ✗  code/codereval
  mt28664_base    partial (6/10)    partial (3/10)
    ✗  code/editbench
    ✗  code/codereval
    ✗  crux/cruxeval_i
    ✗  crux/cruxeval_o

── olmo-3-7b-think ── (file: 67/70, summary: 37/70)
  run             file              summary
  ──────────────  ────────────────  ────────────────
  mt4096_base     complete          partial (9/10)
  mt8192_base     complete          partial (9/10)
  mt12288_base    complete          partial (9/10)
  mt16384_base    complete          partial (9/10)
  mt20480_base    partial (9/10)    partial (1/10)
    ✗  code/co

In [19]:
"""Overall summary across all experiment types."""

total_found = baseline_found + twopass_found
total_all = baseline_total + twopass_total

print("══════════════════════════════")
print("  OVERALL SUMMARY")
print("══════════════════════════════")
print(
    f"  Baseline   {baseline_found:4d} / {baseline_total:4d}  ({baseline_found / baseline_total:.0%})"
    f"  evaluated: {baseline_evaluated}/{baseline_total}"
)
print(
    f"  Two-pass   {twopass_found:4d} / {twopass_total:4d}  ({twopass_found / twopass_total:.0%})"
)
print(
    f"  Total      {total_found:4d} / {total_all:4d}  ({total_found / total_all:.0%})"
)
print("══════════════════════════════")

══════════════════════════════
  OVERALL SUMMARY
══════════════════════════════
  Baseline     30 /   30  (100%)  evaluated: 13/30
  Two-pass    193 /  210  (92%)
  Total       223 /  240  (93%)
══════════════════════════════


In [20]:
"""List any unexpected files present in the output directories (not in the expected set)."""

# build the full set of expected paths for quick lookup
expected_paths: set[Path] = set()

for model in MODELS:
    # baseline files
    for dataset in DATASETS:
        stem = Path(dataset).stem
        expected_paths.add(
            (
                OUTPUT_DIR / "full_baseline" / model / f"{stem}_{CONFIG}_baseline.json"
            ).resolve(),
        )
    # two-pass files
    for tokens in MAX_THINK_TOKENS:
        for suffix in OVERFLOW_SUFFIXES:
            run_name = f"mt{tokens}_{suffix}"
            for dataset in DATASETS:
                stem = Path(dataset).stem
                expected_paths.add(
                    (
                        OUTPUT_DIR
                        / "full_transition"
                        / model
                        / f"{stem}_{CONFIG}_{run_name}.json"
                    ).resolve(),
                )

# scan the output subdirectories for actual .json files
unexpected = []
for subdir in ["full_baseline", "full_transition"]:
    for path in sorted((OUTPUT_DIR / subdir).rglob("*.json")):
        if path.resolve() not in expected_paths:
            unexpected.append(path)

if unexpected:
    print(f"Unexpected files ({len(unexpected)}):")
    for p in unexpected:
        print(f"  {p}")
else:
    print("No unexpected files found.")

No unexpected files found.
